# Exercício 12 — Classificação

Este exercício repete o pipeline da aula na **sua coleta** (a mesma das Aulas 5 a 7 e do Exercício 11). Você vai transformar `plays` num rótulo de "viralizou", treinar um classificador e avaliá-lo com matriz de confusão, não só com acurácia.

Copie a sua `exportacao.csv` para `dados/exportacao.csv` nesta pasta. Antes de tudo, copie a pasta `exercicios/` para dentro da sua pasta de entregas (`extracao-dados-trabalhos-seunome`), numa pasta `12-classificacao` dentro de `projetos/`.

## Preparação do ambiente

Dentro da pasta, no Windows (Prompt de Comando ou Terminal integrado do VS Code):

```cmd
uv venv .venv
uv pip install -r requirements.txt
```

No Mac (Terminal), os mesmos comandos. Se o `uv` não funcionar, `pip install -r requirements.txt` com o ambiente ativado.

## Parte 0 — Dados e a decisão do rótulo

**Fonte dos dados:** Exportação do TikTok via Zeeschuimer, Aula 7.

**Como você vai definir "viralizou" (preencha antes de codar):**

> Vou marcar como 'viralizou' os posts com plays acima do percentil 75 da minha coleta. Desse modo, cerca de 25% dos posts será classificao como viral. Por exemplo, se fosse colocado como viral posts com plays acima do percentil 90, a quantidade de posts marcadas como viral seria extremamente baixa, ainda mais porque a coleta de posts não é tão grande. Usar o percentil 75 seria mais "justo" em uma coleta pequena como está (1138 posts), visto que o objetivo é ver posts que performaram relativamente melhor do que a maioria (nada muito exagerado), e não selecionar apenas os posts que tiveram uma performance de extrema viralização. 

## Parte 1 — Carregar, montar features e o rótulo

Ajuste os nomes de coluna se a sua plataforma for diferente. **Nada de vazamento:** `likes`/`comments`/`shares`/`plays` não entram no `X`.

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import re

df = pd.read_csv("dados/exportacao.csv", sep=";")
df = df.drop_duplicates()
df = df[df["plays"] > 0].copy()

# features (reaproveite/adapte as da aula)
# complete aqui: crie pelo menos TRÊS features (pode copiar do Exercício 11)
X = pd.DataFrame(index=df.index)
X["seguidores_autor"] = df["author_followers"]
X["videos_autor"] = df["author_videos"]
X["tam_legenda"] = df["body"].fillna("").str.len()
padrao_emoji = re.compile("[\U0001F000-\U0001FAFF\u2600-\u27BF]")
X["n_emojis"] = df["body"].fillna("").apply(lambda s: len(padrao_emoji.findall(s)))
X["n_hashtags"] = df["hashtags"].fillna("").apply(lambda s: 0 if s == "" else len(s.split(",")))
momento = pd.to_datetime(df["timestamp"])
X["hora"] = momento.dt.hour
X["dia_semana"] = momento.dt.dayofweek

print("Features:", list(X.columns))

Features: ['seguidores_autor', 'videos_autor', 'tam_legenda', 'n_emojis', 'n_hashtags', 'hora', 'dia_semana']


In [2]:
# complete aqui: crie o rótulo y (0/1) a partir do corte que você definiu na Parte 0
# exemplo com percentil 90:
# corte = df["plays"].quantile(0.90)
# y = (df["plays"] > corte).astype(int)

corte = df["plays"].quantile(0.75)  # o valor de plays que separa os 10% mais vistos
y = (df["plays"] > corte).astype(int)  # 1 = viralizou, 0 = não

print(f"Corte (percentil 75 de plays): {int(corte):,} visualizações")
print(f"Posts marcados como 'viralizou': {y.sum()} de {len(y)}  ({y.mean():.0%})")

Corte (percentil 75 de plays): 688,375 visualizações
Posts marcados como 'viralizou': 285 de 1138  (25%)


## Parte 2 — Treino, teste e regressão logística

In [4]:
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import confusion_matrix, accuracy_score, precision_score, recall_score, f1_score

X_treino, X_teste, y_treino, y_teste = train_test_split(
    X, y, test_size=0.25, random_state=42, stratify=y
)

modelo = LogisticRegression(max_iter=2000)
modelo.fit(X_treino, y_treino)
decisao = modelo.predict(X_teste)
probabilidade = modelo.predict_proba(X_teste)[:, 1]

cm = confusion_matrix(y_teste, decisao)
print("Matriz de confusão [ [VN, FP], [FN, VP] ]:")
print(cm)
print(f"Acurácia {accuracy_score(y_teste, decisao):.2f} | "
      f"Precisão {precision_score(y_teste, decisao, zero_division=0):.2f} | "
      f"Recall {recall_score(y_teste, decisao, zero_division=0):.2f} | "
      f"F1 {f1_score(y_teste, decisao, zero_division=0):.2f}")

Matriz de confusão [ [VN, FP], [FN, VP] ]:
[[211   3]
 [ 61  10]]
Acurácia 0.78 | Precisão 0.77 | Recall 0.14 | F1 0.24


## Parte 3 — Testar dois thresholds

Complete: para pelo menos **dois** cortes diferentes de 0,5 (ex.: 0,30 e 0,20), recalcule precisão, recall e F1 usando `probabilidade`. Imprima os resultados lado a lado com o corte 0,50.

In [5]:
# complete aqui: um laço sobre alguns valores de threshold, aplicando (probabilidade >= t)
for t in [0.50, 0.30, 0.20, 0.15]:
    decisao_t = (probabilidade >= t).astype(int)
    p = precision_score(y_teste, decisao_t, zero_division=0)
    r = recall_score(y_teste, decisao_t, zero_division=0)
    f = f1_score(y_teste, decisao_t, zero_division=0)
    vp = int(((decisao_t == 1) & (y_teste.values == 1)).sum())
    fp = int(((decisao_t == 1) & (y_teste.values == 0)).sum())
    print(f"corte {t:.2f}:  precisão {p:.2f}  recall {r:.2f}  F1 {f:.2f}   (pegou {vp} virais, {fp} alarmes falsos)")

corte 0.50:  precisão 0.77  recall 0.14  F1 0.24   (pegou 10 virais, 3 alarmes falsos)
corte 0.30:  precisão 0.62  recall 0.32  F1 0.43   (pegou 23 virais, 14 alarmes falsos)
corte 0.20:  precisão 0.31  recall 0.83  F1 0.45   (pegou 59 virais, 130 alarmes falsos)
corte 0.15:  precisão 0.27  recall 0.99  F1 0.43   (pegou 70 virais, 188 alarmes falsos)


## Parte 4 — Uma árvore, para comparar

Complete: treine uma `DecisionTreeClassifier(max_depth=4, random_state=42, class_weight="balanced")`, preveja em `X_teste`, calcule precisão/recall/F1 e imprima as `feature_importances_` ordenadas.

In [6]:
from sklearn.tree import DecisionTreeClassifier

arvore = DecisionTreeClassifier(max_depth=4, random_state=42, class_weight="balanced")
arvore.fit(X_treino, y_treino)
decisao_arvore = arvore.predict(X_teste)

print("Árvore (profundidade 4, class_weight balanced):")
print(f"  precisão {precision_score(y_teste, decisao_arvore, zero_division=0):.2f}"
      f"  recall {recall_score(y_teste, decisao_arvore, zero_division=0):.2f}"
      f"  F1 {f1_score(y_teste, decisao_arvore, zero_division=0):.2f}")
print()

importancias = pd.DataFrame({
    "feature": X.columns,
    "importancia": arvore.feature_importances_,
}).sort_values("importancia", ascending=False)
importancias


Árvore (profundidade 4, class_weight balanced):
  precisão 0.48  recall 0.61  F1 0.54



,feature,importancia
0,seguidores_autor,0.704918
1,videos_autor,0.183832
5,hora,0.038685
3,n_emojis,0.033047
6,dia_semana,0.026373
2,tam_legenda,0.013145
4,n_hashtags,0.000000


## Parte 5 — README de reprodução

Crie `README.md` dentro de `projetos/12-classificacao/` na sua pasta de entregas:

**Fonte e período dos dados; quantos posts entraram:**

> Fonte: Exportação do TikTok via Zeeschuimer, Aula 7. Período: 19/08/2026. Posts que entraram: 1138.

**Como você definiu "viralizou", e por quê:**

> Marquei como'viralizou' os posts com plays acima do percentil 75 da minha coleta. Desse modo, cerca de 25% dos posts será classificao como viral. Por exemplo, se fosse colocado como viral posts com plays acima do percentil 90, a quantidade de posts marcadas como viral seria extremamente baixa, ainda mais porque a coleta de posts não é tão grande. Usar o percentil 75 seria mais "justo" em uma coleta pequena como está (1138 posts), visto que o objetivo é ver posts que performaram relativamente melhor do que a maioria (nada muito exagerado), e não selecionar apenas os posts que tiveram uma performance de extrema viralização. 

**Quantos posts ficaram como "viral" (a classe é rara?):**

> Foram 285 de 1138 posts marcados como 'viralizou'(25%). A classe pode ser considerada rara, mas ao analisar o recall (0,14), ele se apresenta baixo, ou seja, o modelo classificou apenas 14% dos posts que realmente eram virais como virais, ignorando 86% dos posts que deveriam ser classficados desse modo também, gerando falsos negativos. Observando a precisão, ela mostra que em 77% das vezes que o modelo marca um post como viral, ele acerta. Ou seja, o modelo se mostra seletivo, visto que só marca viral quando tem bastante confiança naquele resultado, por isso erra pouco (23% das vezes), porém, por ser cauteloso demais, ele deixa passar a maioria dos virais de verdade (recall baixo)

**Features usadas; colunas descartadas por vazamento:**

> Features criadas: 'seguidores_autor', 'videos_autor', 'tam_legenda', 'n_emojis', 'n_hashtags', 'hora', 'dia_semana'. Colunas descartadas por vazamento: "likes", "comments", "shares" e "plays".

**Matriz de confusão do seu melhor modelo, e uma leitura: a favor de quem ele erra?** (deixa passar muitos virais = recall baixo; ou dá muito alarme falso = precisão baixa)

> A matriz de confusão da regressão logística fica assim: VN = 211 / FN = 61 / FP = 3 / VP = 10. Ele está errando a favor dos falsos negativos, ou seja, existem 61 posts que deveriam ser classificados como virais, mas não estão sendo. Isso também pode ser observado por meio do recall de 0.14, ou seja, ele está baixo, deixando passar muitos posts que deveriam estar sendo considerados como virais.

**O que mudou quando você baixou o threshold:**

> Quando abaixei o threshold a precisão tende a cair e o recall tende a aumentar, ou seja, ele começa a pegar mais posts virais (recall), mas começa a errar mais ao classificar um posts como viral (precisão)

**Declaração de uso de IA:** 

> Não usei IA nesta entrega.

## Parte 6 — Conferência final

- [ ] `dados/exportacao.csv` é a sua coleta, e `dados/` está no `.gitignore`.
- [ ] O rótulo "viralizou" foi definido por um corte **justificado** no README.
- [ ] O `X` tem pelo menos três features sem vazamento.
- [ ] A matriz de confusão e precisão/recall/F1 estão calculadas, não só a acurácia.
- [ ] Pelo menos dois thresholds diferentes de 0,5 foram testados.
- [ ] Uma árvore foi treinada e comparada.
- [ ] O README responde todas as perguntas da Parte 5, incluindo a leitura "a favor de quem o modelo erra".
- [ ] O notebook roda do início ao fim com Kernel → Restart e Run All.
- [ ] O README registra o uso de IA (ou informa que não houve).
- [ ] Notebook e README copiados em `projetos/12-classificacao/` na pasta de entregas.
- [ ] Você já fez `git add`, `git commit` e `git push` dessa entrega.